In [1]:
import pandas as pd
import re
df = pd.read_csv("scraped_jobs.csv")
df = df.dropna(subset=["description"])

initial_rows = len(df)
print("Initial dataset size:",initial_rows)

df = df.drop_duplicates(
    subset=["role", "description"],
    keep="first"
)


def has_valid_words(text, min_words=10):
    words = re.findall(r"[a-zA-Z]{2,}", text)
    return len(words) >= min_words

df = df[df["description"].apply(has_valid_words)]

len(df)



df["desc_length"] = df["description"].str.len()
avg_length = df["desc_length"].mean()
MIN_LENGTH = avg_length * 0.4
df = df[df["desc_length"] >= MIN_LENGTH]

def valid_job_title(title):
    if pd.isna(title):
        return False
    title = title.lower()
    if len(title) < 3:
        return False
    if not re.search(r"[a-zA-Z]", title):
        return False
    return True

df = df[df["role"].apply(valid_job_title)]

print(f"Final dataset size: {len(df)} job postings")

Initial dataset size: 954
Final dataset size: 764 job postings


In [2]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"<.*?>", " ", text)      # remove html
    text = re.sub(r"[^a-zA-Z\s]", " ", text) # remove punctuation/numbers
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_description"] = df["description"].apply(clean_text)

import spacy
nlp = spacy.load("en_core_web_sm")

def lemmatize(text):
    doc = nlp(text)
    return [token.lemma_ for token in doc
            if not token.is_stop and token.is_alpha]

df["tokens"] = df["clean_description"].apply(lemmatize)

In [3]:
def clean_job_title(title):
    title = title.lower()
    title = re.sub(r"@.*", "", title)                # remove company
    title = re.sub(r"\(.*?\)", "", title)            # remove brackets
    title = re.sub(r"[-/|]", " ", title)              # separators
    title = re.sub(r"\b(senior|jr|junior|lead|tech lead|principal)\b", "", title)
    title = re.sub(r"\s+", " ", title).strip()
    return title

df["clean_title"] = df["role"].apply(clean_job_title)

ROLE_KEYWORDS = {
    # --- Engineering & Development ---
    "software engineer": ["software engineer", "software developer", "swe", "developer", "application developer", "systems programmer"],
    "web developer": ["web developer", "frontend developer", "backend developer", "full stack developer", "ui developer", "javascript developer"],
    "mobile developer": ["mobile developer", "android developer", "ios developer", "react native developer", "flutter developer", "swift developer"],
    "game developer": ["game developer", "unity developer", "unreal engine developer", "game programmer", "graphics engineer"],
    "embedded engineer": ["embedded engineer", "firmware engineer", "iot developer", "hardware engineer", "systems engineer"],

    # --- Data & AI ---
    "data scientist": ["data scientist", "machine learning engineer", "ml engineer", "ai engineer", "nlp engineer", "computer vision engineer", "deep learning engineer"],
    "data analyst": ["data analyst", "business analyst", "bi analyst", "product analyst", "data visualizer", "tableau developer"],
    "data engineer": ["data engineer", "etl developer", "big data engineer", "data architect", "analytics engineer"],
    "database administrator": ["dba", "database administrator", "database engineer", "sql developer"],

    # --- Infrastructure, Cloud & DevOps ---
    "devops engineer": ["devops engineer", "site reliability engineer", "sre", "platform engineer", "automation engineer", "build engineer"],
    "cloud engineer": ["cloud engineer", "cloud architect", "aws architect", "azure engineer", "gcp architect", "cloud consultant"],
    "network engineer": ["network engineer", "network architect", "systems administrator", "sysadmin", "infrastructure engineer"],

    # --- Security ---
    "cybersecurity engineer": ["security engineer", "cybersecurity", "information security", "soc analyst", "pentester", "ethical hacker", "application security engineer", "security architect"],
    "compliance officer": ["it compliance", "grc analyst", "it auditor", "privacy engineer"],

    # --- Quality & Testing ---
    "qa engineer": ["qa engineer", "quality assurance", "software tester", "automation tester", "sdet", "manual tester", "performance engineer"],

    # --- Product & Management ---
    "product manager": ["product manager", "pm", "technical product manager", "product owner"],
    "project manager": ["project manager", "it project manager", "scrum master", "agile coach", "delivery manager"],
    "it manager": ["it manager", "cto", "cio", "engineering manager", "vpe", "it director"],

    # --- Design & UX ---
    "ux designer": ["ux designer", "ui designer", "product designer", "user researcher", "interaction designer", "ux writer"],

    # --- Specialized Platforms & ERP ---
    "salesforce developer": ["salesforce developer", "sfdc developer", "salesforce admin", "crm developer"],
    "erp consultant": ["sap consultant", "oracle functional consultant", "dynamics 365 developer", "erp analyst"],
    "service management": ["itsm manager", "servicenow developer", "itil consultant"]
}

def normalize_role(title):
    for canonical_role, keywords in ROLE_KEYWORDS.items():
        for kw in keywords:
            if kw in title:
                return canonical_role
    return "other"

df["normalized_role"] = df["clean_title"].apply(normalize_role)
df

,role,description,desc_length,clean_description,tokens,clean_title,normalized_role
0,Software Developer,Software Developer\r\nDiligent Consulting Grou...,1547,software developer diligent consulting group p...,"[software, developer, diligent, consulting, gr...",software developer,software engineer
1,Software Engineer,Software Engineer\r\nHayleys\r\n| Â 2024-06-01...,1424,software engineer hayleys similar jobs apply a...,"[software, engineer, hayley, similar, job, app...",software engineer,software engineer
2,Senior Software Engineer - Next.js/React.js @i...,Senior Software Engineer - Next.js/React.js @i...,2377,senior software engineer next js react js ilab...,"[senior, software, engineer, js, react, js, il...",software engineer next.js react.js,software engineer
3,Senior Tech Lead - Software Engineering,Senior Tech Lead - Software Engineering\r\nDia...,1982,senior tech lead software engineering dialog s...,"[senior, tech, lead, software, engineering, di...",software engineering,software engineer
4,Senior Software Engineer,Senior Software Engineer\r\nHayleys\r\n| Â 202...,1370,senior software engineer hayleys similar jobs ...,"[senior, software, engineer, hayley, similar, ...",software engineer,software engineer
...,...,...,...,...,...,...,...
944,Graphic Designer,CA .\r\nF PO) HOME ABOUT US PROFILE CONTACT US...,1556,ca f po home about us profile contact us ss in...,"[f, po, home, profile, contact, ss, industrial...",graphic designer,other
948,Application Support Associate,K\r\nZeeks Lab\r\nAre you passionate about\r\n...,2225,k zeeks lab are you passionate about technolog...,"[k, zeeks, lab, passionate, technology, custom...",application support associate,other
949,UI-UX Developer,"At eBEYONDS, an International eBusiness & Digi...",1630,at ebeyonds an international ebusiness digital...,"[ebeyond, international, ebusiness, digital, m...",ui ux developer,software engineer
953,Consultant (SAP FICO) (1),"John Keells Information Technology (Pvt} Ltd, ...",2394,john keells information technology pvt ltd the...,"[john, keells, information, technology, pvt, l...",consultant,other


In [4]:
!pip install --upgrade transformers peft datasets seqeval accelerate


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
SKILL_DICTIONARY = [
    # Programming Languages
    "python", "java", "c", "c++", "c#", "javascript", "typescript", "go", "rust",
    "php", "ruby", "swift", "kotlin", "r", "matlab", "scala", "perl", "bash",
    "powershell", "objective-c", "groovy", "dart", "lua", "haskell",

    # Web Development
    "html", "css", "sass", "less", "bootstrap", "tailwind css",
    "react", "angular", "vue", "next.js", "nuxt.js", "svelte",
    "node.js", "express.js", "nestjs",
    "django", "flask", "fastapi",
    "spring", "spring boot",
    "laravel", "codeigniter",
    "asp.net", "asp.net core",
    "graphql", "rest api", "soap",

    # Databases
    "mysql", "postgresql", "oracle", "sql server", "sqlite",
    "mongodb", "cassandra", "couchdb", "redis", "dynamodb",
    "firebase", "neo4j", "elasticsearch",
    "nosql", "sql", "pl/sql",

    # Cloud & DevOps
    "aws", "azure", "google cloud", "gcp",
    "ec2", "s3", "lambda", "cloudformation",
    "docker", "kubernetes", "helm",
    "terraform", "ansible", "chef", "puppet",
    "jenkins", "gitlab ci", "github actions", "circleci",
    "linux", "unix",
    "nginx", "apache",
    "devops", "site reliability engineering", "sre",

    # Data Science & Machine Learning
    "machine learning", "deep learning", "artificial intelligence",
    "natural language processing", "nlp", "computer vision",
    "data science", "data analysis", "data engineering",
    "pandas", "numpy", "scipy", "scikit-learn",
    "tensorflow", "keras", "pytorch",
    "xgboost", "lightgbm",
    "opencv",
    "statistics", "linear regression", "logistic regression",
    "clustering", "classification", "time series",

    # Big Data
    "hadoop", "spark", "pyspark", "kafka", "flink",
    "hive", "pig", "hbase", "airflow",
    "data warehousing", "etl",

    # Mobile Development
    "android", "ios",
    "react native", "flutter", "xamarin",
    "android studio", "xcode",

    # Cybersecurity
    "cybersecurity", "information security",
    "penetration testing", "ethical hacking",
    "network security", "application security",
    "cryptography", "siem", "soc",
    "firewalls", "ids", "ips",
    "owasp", "iam",

    # Networking
    "tcp/ip", "udp", "dns", "dhcp",
    "http", "https",
    "routing", "switching",
    "vpn", "lan", "wan",
    "ccna", "ccnp",

    # Operating Systems
    "windows", "linux", "macos",
    "red hat", "ubuntu", "debian", "centos",

    # Software Engineering
    "object oriented programming", "oop",
    "design patterns",
    "clean code", "solid principles",
    "data structures", "algorithms",
    "microservices", "monolithic architecture",
    "event driven architecture",

    # Testing & QA
    "unit testing", "integration testing", "system testing",
    "selenium", "cypress", "playwright",
    "junit", "pytest", "testng",
    "automation testing", "manual testing",

    # Version Control & Tools
    "git", "github", "gitlab", "bitbucket",
    "jira", "confluence",
    "postman", "swagger",

    # UI / UX
    "figma", "adobe xd", "sketch",
    "ui design", "ux design",
    "wireframing", "prototyping",

    # ERP / CRM / Enterprise
    "sap", "oracle erp", "salesforce",
    "workday", "servicenow",

    # Methodologies
    "agile", "scrum", "kanban",
    "waterfall", "devsecops",

    # Misc / Emerging
    "blockchain", "web3", "smart contracts",
    "solidity",
    "internet of things", "iot",
    "robotic process automation", "rpa",
    "computer graphics", "game development",
    "unity", "unreal engine"
]

SKILL_SET = set([skill.lower() for skill in SKILL_DICTIONARY])

In [6]:
def normalize(text):
    text = text.lower()
    text = text.replace("-", " ")
    text = text.replace(".", " ")
    text = text.replace("nodejs", "node js")
    text = text.replace("reactjs", "react")
    return text


def auto_label(text):

    text = normalize(text)

    words = text.split()
    labels = ["O"] * len(words)

    for skill in SKILL_SET:

        skill_tokens = skill.split()
        skill_len = len(skill_tokens)

        for i in range(len(words) - skill_len + 1):

            phrase = " ".join(words[i:i+skill_len])
            pattern = r"\b" + re.escape(skill) + r"\b"

            if re.search(pattern, phrase):

                labels[i] = "B-SKILL"

                for j in range(1, skill_len):
                    labels[i+j] = "I-SKILL"

    return words, labels   # <-- FIX


labeled_data = []

for text in df["clean_description"]:

    tokens, tags = auto_label(text)

    labeled_data.append({
        "tokens": tokens,
        "ner_tags": tags
    })

In [7]:
label_list = ["O", "B-SKILL", "I-SKILL"]
label_to_id = {l: i for i, l in enumerate(label_list)}
id_to_label = {i: l for l, i in label_to_id.items()}

for item in labeled_data:
    item["ner_tags"] = [label_to_id[tag] for tag in item["ner_tags"]]


from datasets import Dataset

dataset = Dataset.from_list(labeled_data)
dataset = dataset.train_test_split(test_size=0.1)

C:\Users\yasit\PycharmProjects\PythonProject\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []

    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(label[word_idx] if label[word_idx] == 2 else -100)

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)

Map: 100%|██████████| 77/77 [00:00<00:00, 171.11 examples/s]


In [9]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(label_list),
    id2label=id_to_label,
    label2id=label_to_id
)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1515.11it/s]
DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
import numpy as np
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score
from transformers import TrainingArguments, Trainer, DataCollatorForTokenClassification

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_labels = []
    true_predictions = []

    for pred, lab in zip(predictions, labels):
        true_labels.append([id_to_label[l] for l in lab if l != -100])
        true_predictions.append([
            id_to_label[p] for (p, l) in zip(pred, lab) if l != -100
        ])

    return {
        "precision": precision_score(true_labels, true_predictions),
        "recall": recall_score(true_labels, true_predictions),
        "f1": f1_score(true_labels, true_predictions)
    }


data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="./skill_bert",
    eval_strategy="epoch",
    save_strategy="epoch",

    logging_strategy="steps",
    logging_steps=50,

    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_dir="./logs",
    load_best_model_at_end=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
C:\Users\yasit\PycharmProjects\PythonProject\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


In [21]:
# Use a new directory name to bypass the Windows file lock
save_path = "./skill_bert_final"

# Save the model
trainer.save_model(save_path)

# Explicitly save tokenizer with all necessary files
tokenizer.save_pretrained(save_path, legacy_format=False)

# Also save the label mappings for consistency
import json
label_config = {
    "id2label": id_to_label,
    "label2id": label_to_id,
    "label_list": label_list
}

with open(f"{save_path}/label_config.json", "w") as f:
    json.dump(label_config, f)

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.23it/s]


In [22]:
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification

model_path = "./skill_bert_final"

# Load with explicit parameters to ensure consistency
tokenizer = AutoTokenizer.from_pretrained(
    model_path,
    use_fast=False,  # Use slow tokenizer for consistency
    trust_remote_code=True
)

model = AutoModelForTokenClassification.from_pretrained(
    model_path,
    trust_remote_code=True,
    local_files_only=False  # Allow fallback to original model if needed
)

# Verify compatibility
print(f"Tokenizer vocab size: {len(tokenizer)}")
print(f"Model vocab size: {model.config.vocab_size}")

if len(tokenizer) != model.config.vocab_size:
    print("⚠️ WARNING: Vocabulary size mismatch detected!")
    print("Reloading with base model tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained("distilbert-base-cased")

model.eval()


Loading weights: 100%|██████████| 102/102 [00:00<00:00, 2248.59it/s]

Tokenizer vocab size: 30522
Model vocab size: 30522


DistilBertForTokenClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
   

In [23]:
from transformers import pipeline

# 1. Initialize the pipeline WITHOUT aggregation to get raw, precise subword offsets
ner_pipe = pipeline(
    "ner",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="none",  # Change to none so we can manually stitch the broken tags
    device=0 if torch.cuda.is_available() else -1
)

def extract_skills(text):
    try:
        entities = ner_pipe(text)
        final_skills = []

        for ent in entities:
            # Only look at tokens the model flagged as SKILL
            if 'SKILL' in ent['entity']:
                # Strip the ## from the word
                word = ent['word'].replace('##', '')
                start = ent['start']
                end = ent['end']

                # If we have a previous skill, check if this new piece physically touches it in the text
                # (e.g. 'ko' ends at character 12, '##tlin' starts at character 12)
                if final_skills and start == final_skills[-1]['end']:
                    final_skills[-1]['word'] += word
                    final_skills[-1]['end'] = end
                else:
                    # Otherwise, it's a brand new skill
                    final_skills.append({'word': word, 'start': start, 'end': end})

        # Extract just the words, remove duplicates, and drop tiny 1-letter hallucinations
        skills = list(set([s['word'].strip() for s in final_skills if len(s['word'].strip()) > 1]))
        return skills

    except Exception as e:
        print(f"Error in extraction: {e}")
        return []

# Apply the perfected function to your dataframe
df["clean_description2"] = df["clean_description"]
df["extracted_skills"] = df["clean_description2"].apply(extract_skills)

# Preview the clean results
df

,role,description,desc_length,clean_description,tokens,clean_title,normalized_role,clean_description2,extracted_skills
0,Software Developer,Software Developer\nDiligent Consulting Group ...,1513,software developer diligent consulting group p...,"[software, developer, diligent, consulting, gr...",software developer,software engineer,software developer diligent consulting group p...,"[native, android, react, jira, github, ios, sql]"
1,Software Engineer,Software Engineer\nHayleys\n| Â 2024-06-01\nSi...,1400,software engineer hayleys similar jobs apply a...,"[software, engineer, hayley, similar, job, app...",software engineer,software engineer,software engineer hayleys similar jobs apply a...,"[vu, angular, design, react, html, cs, javascr..."
2,Senior Software Engineer - Next.js/React.js @i...,Senior Software Engineer - Next.js/React.js @i...,2340,senior software engineer next js react js ilab...,"[senior, software, engineer, js, react, js, il...",software engineer next.js react.js,software engineer,senior software engineer next js react js ilab...,"[spring, jenkins, graph, react, javascript, aw..."
3,Senior Tech Lead - Software Engineering,Senior Tech Lead - Software Engineering\nDialo...,1956,senior tech lead software engineering dialog s...,"[senior, tech, lead, software, engineering, di...",software engineering,software engineer,senior tech lead software engineering dialog s...,"[gi, aw]"
4,Senior Software Engineer,Senior Software Engineer\nHayleys\n| Â 2024-11...,1346,senior software engineer hayleys similar jobs ...,"[senior, software, engineer, hayley, similar, ...",software engineer,software engineer,senior software engineer hayleys similar jobs ...,"[vu, angular, design, react, html, cs, javascr..."
...,...,...,...,...,...,...,...,...,...
944,Graphic Designer,CA .\nF PO) HOME ABOUT US PROFILE CONTACT US\n...,1529,ca f po home about us profile contact us ss in...,"[f, po, home, profile, contact, ss, industrial...",graphic designer,other,ca f po home about us profile contact us ss in...,[]
948,Application Support Associate,K\nZeeks Lab\nAre you passionate about\nTechno...,2188,k zeeks lab are you passionate about technolog...,"[k, zeeks, lab, passionate, technology, custom...",application support associate,other,k zeeks lab are you passionate about technolog...,"[security, information]"
949,UI-UX Developer,"At eBEYONDS, an International eBusiness & Digi...",1601,at ebeyonds an international ebusiness digital...,"[ebeyond, international, ebusiness, digital, m...",ui ux developer,software engineer,at ebeyonds an international ebusiness digital...,"[react, cs, typescript, fig, java, gi, vu]"
953,Consultant (SAP FICO) (1),"John Keells Information Technology (Pvt} Ltd, ...",2366,john keells information technology pvt ltd the...,"[john, keells, information, technology, pvt, l...",consultant,other,john keells information technology pvt ltd the...,"[sales, sap]"


In [4]:
import matplotlib.pyplot as plt
import pandas as pd

log_history = trainer.state.log_history
df = pd.DataFrame(log_history)
print(df.columns)

# Evaluation metrics
eval_df = df[df["eval_loss"].notna()]

plt.figure(figsize=(8,5))
plt.plot(eval_df["epoch"], eval_df["eval_precision"], label="Precision")
plt.plot(eval_df["epoch"], eval_df["eval_recall"], label="Recall")
plt.plot(eval_df["epoch"], eval_df["eval_f1"], label="F1 Score")
plt.xlabel("Epoch")
plt.ylabel("Score")
plt.title("Model Performance Metrics")
plt.legend()
plt.show()

# Training loss (correct column name)
train_df = df[df["loss"].notna()]

plt.figure(figsize=(8,5))
plt.plot(train_df["step"], train_df["loss"], label="Training Loss")
plt.xlabel("Training Step")
plt.ylabel("Loss")
plt.title("Training Loss Curve")
plt.legend()
plt.show()

# Validation loss
plt.figure(figsize=(8,5))
plt.plot(eval_df["epoch"], eval_df["eval_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Validation Loss Curve")
plt.legend()
plt.show()

# Training vs Validation loss
import matplotlib.pyplot as plt
import pandas as pd

log_history = trainer.state.log_history
df = pd.DataFrame(log_history)
# Training loss
train_df = df[df["loss"].notna()]
# Validation loss
eval_df = df[df["eval_loss"].notna()]
plt.figure(figsize=(8,5))
plt.plot(train_df["epoch"], train_df["loss"], label="Training Loss")
plt.plot(eval_df["epoch"], eval_df["eval_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.show()

NameError: name 'trainer' is not defined

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix


predictions, labels, _ = trainer.predict(tokenized_datasets["test"])
predictions = np.argmax(predictions, axis=2)
true_labels = []
pred_labels = []

for pred, lab in zip(predictions, labels):
    true = [id_to_label[l] for l in lab if l != -100]
    pred = [id_to_label[p] for (p, l) in zip(pred, lab) if l != -100]
    true_labels.append(true)
    pred_labels.append(pred)

y_true = [label for seq in true_labels for label in seq]
y_pred = [label for seq in pred_labels for label in seq]

label_names = ["B-SKILL", "I-SKILL", "O"]


cm = confusion_matrix(y_true, y_pred, labels=label_names)
plt.figure(figsize=(6,5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=label_names,
    yticklabels=label_names
)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix – BERT Skill Extraction Model")
plt.show()

from seqeval.metrics import classification_report

report = classification_report(true_labels, pred_labels)
print(report)

NameError: name 'trainer' is not defined